In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("student-mat.csv", sep=";");
df.head()
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

,0
school,0
sex,0
age,0
address,0
famsize,0
Pstatus,0
Medu,0
Fedu,0
Mjob,0
Fjob,0


In [ ]:
df=pd.get_dummies(df, drop_first=True)
df = df.astype(int)
df.head()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [ ]:
df['risk']=df['G3'].apply(lambda x: 1 if x<10 else 0)
df=df.drop(['G1','G2','G3'], axis=1)
df.head()


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes,risk
0,18,4,4,2,2,0,4,3,4,1,...,0,1,0,0,0,1,1,0,0,1
1,17,1,1,1,2,0,5,3,3,1,...,0,0,1,0,0,0,1,1,0,1
2,15,1,1,1,2,3,4,3,2,2,...,0,1,0,1,0,1,1,1,0,0
3,15,4,2,1,3,0,3,2,2,1,...,0,0,1,1,1,1,1,1,1,0
4,16,3,3,1,2,0,4,3,2,1,...,0,0,1,1,0,1,1,0,0,0


In [ ]:
X=df.drop('risk', axis=1)
y=df['risk']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred= model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7341772151898734


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
rf_pred=rf_model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score
print("Accuracy",accuracy_score(y_test,rf_pred))

Accuracy 0.7088607594936709


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.71      0.94      0.81        52
           1       0.70      0.26      0.38        27

    accuracy                           0.71        79
   macro avg       0.71      0.60      0.59        79
weighted avg       0.71      0.71      0.66        79



In [ ]:
import pandas as pd

feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importance.sort_values(ascending=False).head(10)

,0
absences,0.080089
failures,0.063530
age,0.054686
goout,0.053396
freetime,0.047047
health,0.047008
Fedu,0.044426
Medu,0.042349
Walc,0.041241
famrel,0.036613


In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
xgb_model=XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train,y_train)
xgb_pred=xgb_model.predict(X_test)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:07:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
from sklearn.metrics import accuracy_score
print("Accuracy for XGBoost:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test,xgb_pred) )

Accuracy for XGBoost: 0.7215189873417721
              precision    recall  f1-score   support

           0       0.76      0.85      0.80        52
           1       0.62      0.48      0.54        27

    accuracy                           0.72        79
   macro avg       0.69      0.66      0.67        79
weighted avg       0.71      0.72      0.71        79

